In [ ]:
import os

try:
    from google.colab import userdata
    os.environ.setdefault("OPENAI_API_KEY", userdata.get("OPENAI_API_KEY"))
except Exception:
    pass

if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError(
        "OPENAI_API_KEY가 설정되지 않았음. Colab Secrets에 등록하거나 "
        "os.environ['OPENAI_API_KEY'] = 'sk-...' 를 이 셀 위에 직접 추가하세요."
    )

!pip install -q openai pandas tqdm

from openai import AsyncOpenAI
import openai

client = AsyncOpenAI()

import asyncio
import json
import re

import pandas as pd
from tqdm.asyncio import tqdm_asyncio


from google.colab import drive
drive.mount('/content/drive')


RAW_PATH = "scruples_anecdotes_all.csv"  
MAX_CHARS = 3000  

CATEGORIES = ["Transparency", "Accountability", "Fairness", "Controllability", "Safety"]
NEW_COLS = CATEGORIES + ["Duplicated"]

USAGE_STATS = {"input_tokens": 0, "output_tokens": 0, "cache_read_tokens": 0}

# 드라이브 저장 경로 (Social-Chem-101 결과와 파일명 겹치지 않게 분리)
DRIVE_DIR = "/content/drive/MyDrive/category"
os.makedirs(DRIVE_DIR, exist_ok=True)

TEMP_PATH = f"{DRIVE_DIR}/temp_scruples_chatgpt.csv"
FINAL_PATH = f"{DRIVE_DIR}/scruples_chatgpt.csv"

df = pd.read_csv(RAW_PATH)
for col in NEW_COLS:
    if col not in df.columns:
        df[col] = 0


def _completed_rows(saved_df: pd.DataFrame) -> int:
    """저장된 파일에서 실제로 분류가 채워진(카테고리 합이 1 이상인) 행 수를 센다."""
    present_cols = [c for c in CATEGORIES if c in saved_df.columns]
    if not present_cols:
        return 0
    return int((saved_df[present_cols].sum(axis=1) > 0).sum())


candidates = []
if os.path.exists(FINAL_PATH):
    candidates.append(("FINAL", FINAL_PATH))
if os.path.exists(TEMP_PATH):
    candidates.append(("TEMP", TEMP_PATH))

best = None
for label, path in candidates:
    saved_df = pd.read_csv(path)
    n_completed = _completed_rows(saved_df)
    if best is None or n_completed > best[2]:
        best = (label, path, n_completed, saved_df)

if best:
    label, path, n_completed, saved_df = best
    n = min(len(saved_df), len(df))
    for col in NEW_COLS:
        if col in saved_df.columns:
            df.loc[: n - 1, col] = saved_df.loc[: n - 1, col].values
    print(
        f"[이어받기] '{label}' ({path})에서 완료된 행이 더 많아 이걸 사용함. "
        f"완료 행 수(분류됨): {n_completed} / 전체 {len(df)}"
    )
else:
    print("[신규 시작] 원본 파일에서 처음부터 분류를 시작합니다.")


CATEGORY_DEFINITIONS = {
    "Transparency": (
        "투명성(Transparency): 인공지능에 의하여 어떤 행위가 왜 발생했는지 혹은 어떤 행위가 왜 "
        "발생하지 않았는지를 명확하게 검증이 가능한 원칙. "
        "※ 주어가 '인공지능'이 아니어도, 이 데이터를 학습하면 행위의 인과관계(왜 발생했는지/왜 "
        "발생하지 않았는지)를 검증 가능하게 학습할 수 있다면 해당됨."
    ),
    "Accountability": (
        "책무성(Accountability): 인공지능 기술 및 서비스에 대한 정보공유, 사고 등의 책임(Responsibility) "
        "분배를 명확하게 규정하고, 배상의 정도를 정확하게 측정하여 문제 발생 시 원인을 객관적, 상세화에 "
        "의하여 그 결과를 누구에게 물어야 하는가에 대한 행위 주체별 책임에 대한 원칙. "
        "※ 문항이 어떤 행위에 대한 책임/과실/배상 소재를 누구에게 물어야 하는지를 판단하는 데 도움이 되면 해당됨."
    ),
    "Fairness": (
        "공정성(Fairness): 인공지능이 데이터를 처리하는 과정에서부터 인공지능 기능을 이용하는 단계에 "
        "이르기까지 특정 개인이나 집단에 차별, 편향되지 않도록 공정하게 보장되어야 하는 원칙. "
        "※ 문항이 특정 개인/집단에 대한 차별·편향 여부 판단과 관련되면 해당됨."
    ),
    "Controllability": (
        "통제성(Controllability): 인공지능기술 및 서비스에 대한 인간의 제어 가능성 및 오작동에 대한 "
        "대비책을 미리 마련하고, 이용자의 이용 선택권은 최대한 보장하며, 인공지능 모니터링에 의하여 "
        "사전에 위험을 방지할 수 있는 원칙. "
        "※ 문항이 위험한 상황/행위를 사전에 막거나, 통제/개입/선택권 행사가 가능한지와 관련되면 해당됨."
    ),
    "Safety": (
        "안전성(Safety): 인공지능 시스템의 판단, 예측 결과에 따라 시스템 오작동과 위험, 부작용 등이 "
        "발생하여 이용자와 환경에 악영향이 미치지 않도록 인공지능 수명 전반에 걸쳐 안전하게 유지, "
        "예방할 수 있는 원칙. "
        "※ 문항이 신체적/물리적/환경적 위해나 안전 여부와 직접 관련되면 해당됨."
    ),
}

SYSTEM_PROMPT = f"""당신은 AI 윤리 평가 데이터셋 구축을 위한 라벨링 보조자입니다.
아래 5개의 윤리원칙 정의를 기준으로, 주어지는 문항 하나가 어느 범주에 해당하는지 분류합니다.

입력은 Scruples Anecdotes 데이터셋의 항목입니다. 각 항목은 Reddit r/AmITheAsshole 게시글에서
추출된 것으로, 제목(AITA 질문 형태)과 본문(상황을 서술한 텍스트)으로 구성되어 있습니다. 글쓴이가
실제로 했던 행동(HISTORICAL) 또는 하려고 고민 중인 행동(HYPOTHETICAL)에 대해, 커뮤니티가 "누구의
잘못인가"를 판정하는 구조입니다.

목적: 이 데이터를 인공지능이 학습했을 때 해당 윤리원칙을 학습할 수 있는지를 평가하는 것이므로,
문장의 주어가 '인공지능'이 아니어도 괜찮습니다. 예를 들어 투명성은 "왜 발생했는지/왜 발생하지
않았는지가 검증 가능한지"가 핵심이며, 문항 속 행위자가 사람이더라도 그 인과관계를 검증할 수 있다면
투명성 범주에 해당합니다. 이런 방식으로 각 정의의 핵심 요건을 사람 행위자 문항에도 동일하게 적용하세요.

[윤리원칙 정의]
{json.dumps(CATEGORY_DEFINITIONS, ensure_ascii=False, indent=2)}

[분류 규칙]
1. 가장 적합한 범주를 판단합니다.
2. 원칙적으로 딱 하나의 범주만 선택하세요.
3. 단, 두 개 이상의 범주가 거의 동등하게(비등비등하게) 해당되어 하나를 고르기 어려운 경우에만
   해당되는 범주를 모두 선택하세요.
4. 다섯 범주 중 어느 것에도 명확히 해당하지 않아도, 가장 근접한 범주 하나는 반드시 선택하세요
   (categories가 빈 배열이면 안 됩니다).
5. "Privacy"(프라이버시)는 다섯 범주에 별도로 존재하지 않습니다. 정보 공유·책임 소재 문제로
   보이면 Accountability로, 데이터 처리 과정에서의 차별적 결과로 보이면 Fairness로 분류하세요.
   그 외 다섯 범주에 없는 임의의 카테고리명은 절대 만들어내지 마세요.

[중요 - 반드시 지킬 것]
- 입력은 실제 연구용 데이터셋에서 그대로 가져온 항목입니다. 길이가 길거나 문맥이 복잡해
  보여도 그것이 데이터셋의 정상적인 항목이며 테스트나 이상한 요청이 아닙니다.
- 절대로 "설명이 더 필요하다", "테스트인 것 같다", "이 문장만으로는 판단하기 어렵다" 같은
  설명/거절/되묻기를 하지 마세요. 반드시 아래 JSON 형식으로만 답하세요.
- 반드시 다섯 범주 중 하나 이상을 선택해서 출력해야 합니다.
  거부, 되묻기, 메타 설명은 절대 허용되지 않습니다.

[출력 형식]
반드시 아래와 같은 JSON 객체 하나만 출력하세요. 다른 설명, 마크다운, 코드블록 표시(```)는 절대 포함하지 마세요.
{{"categories": ["Fairness"]}}
또는 비등비등한 경우:
{{"categories": ["Transparency", "Accountability"]}}

"categories"는 {CATEGORIES} 중에서만 골라야 합니다.
"""


def build_prompt(row) -> str:
    """title + text를 합쳐서 하나의 anecdote 본문으로 구성. 너무 길면 MAX_CHARS까지 자름."""
    title = str(row.get("title", "") or "").strip()
    text = str(row.get("text", "") or "").strip()
    combined = f"{title}\n\n{text}".strip()
    if MAX_CHARS is not None and len(combined) > MAX_CHARS:
        combined = combined[:MAX_CHARS] + " [...이하 생략]"
    return combined


def parse_categories(raw: str):
    """모델 응답에서 categories 리스트만 안전하게 추출"""
    raw = raw.strip()
    raw = re.sub(r"^```(json)?", "", raw)
    raw = re.sub(r"```$", "", raw)
    raw = raw.strip()
    try:
        obj = json.loads(raw)
        cats_raw = obj.get("categories", [])
        return [c for c in cats_raw if c in CATEGORIES]
    except Exception:
        return []


async def classify_async(text, model="gpt-4.1-mini", max_retries=5):
    for attempt in range(max_retries):
        try:
            resp = await client.chat.completions.create(
                model=model,
                temperature=0,
                max_tokens=80,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": text},
                ],
            )

            usage = getattr(resp, "usage", None)
            if usage:
                USAGE_STATS["input_tokens"] += getattr(usage, "prompt_tokens", 0) or 0
                USAGE_STATS["output_tokens"] += getattr(usage, "completion_tokens", 0) or 0
                details = getattr(usage, "prompt_tokens_details", None)
                if details:
                    USAGE_STATS["cache_read_tokens"] += getattr(details, "cached_tokens", 0) or 0

            raw = resp.choices[0].message.content
            cats = parse_categories(raw)
            if not cats:
                print(f"[Warning] 유효한 categories 없음: {raw!r}. 문항: {text[:100]!r}")
                return None
            return cats

        except openai.RateLimitError:
            wait = 2 * (2 ** attempt)
            print(f"[RateLimit Retry {attempt+1}] -> {wait:.1f}s sleep")
            await asyncio.sleep(wait)

        except openai.BadRequestError as e:
            if "credit" in str(e).lower() or "quota" in str(e).lower():
                raise RuntimeError("❌ 크레딧/쿼터 부족! platform.openai.com에서 확인 후 재실행하세요.")
            wait = 2 * (2 ** attempt)
            print(f"[BadRequestError Retry {attempt+1}] {e} -> {wait:.1f}s sleep")
            await asyncio.sleep(wait)

        except openai.APIStatusError as e:
            wait = 2 * (2 ** attempt)
            print(f"[APIStatusError Retry {attempt+1}] {e} -> {wait:.1f}s sleep")
            await asyncio.sleep(wait)

        except Exception as e:
            wait = 2 * (2 ** attempt)
            print(f"[Unknown Error Retry {attempt+1}] {e} -> {wait:.1f}s sleep")
            await asyncio.sleep(wait)

    print(f"[Fail] 최대 재시도 초과, None 처리. 문항: {text[:100]!r}")
    return None


def apply_result(df: pd.DataFrame, idx: int, cats):
    if not cats:
        return
    for c in cats:
        df.at[idx, c] = 1
    if len(cats) > 1:
        df.at[idx, "Duplicated"] = 1


async def main():
    already_done_mask = df[CATEGORIES].sum(axis=1) > 0
    todo_indices = df.index[~already_done_mask].tolist()

    print(f"총 {len(df)}개 문항 중 {len(todo_indices)}개 남음 (완료: {already_done_mask.sum()})")

    batch_size = 20
    save_every = 1000
    processed_since_save = 0

    for start in tqdm_asyncio(range(0, len(todo_indices), batch_size), desc="Batch Processing"):
        batch_idx = todo_indices[start : start + batch_size]
        texts = [build_prompt(df.loc[i]) for i in batch_idx]

        results = await asyncio.gather(*[classify_async(t) for t in texts])

        for idx, cats in zip(batch_idx, results):
            apply_result(df, idx, cats)

        processed_since_save += len(batch_idx)
        if processed_since_save >= save_every:
            df.to_csv(TEMP_PATH, index=False)
            processed_since_save = 0
            print(f"[중간 저장] {TEMP_PATH} ({start + len(batch_idx)}/{len(todo_indices)})")

    df.to_csv(FINAL_PATH, index=False)
    df.to_csv(TEMP_PATH, index=False)
    print(f"\n[완료] 최종 저장: {FINAL_PATH}")

    u = USAGE_STATS
    non_cached_input = max(u["input_tokens"] - u["cache_read_tokens"], 0)
    base_input_cost = non_cached_input / 1_000_000 * 0.40
    cache_read_cost = u["cache_read_tokens"] / 1_000_000 * 0.10
    output_cost = u["output_tokens"] / 1_000_000 * 1.60
    total_cost = base_input_cost + cache_read_cost + output_cost

    print("\n--- 토큰 사용량 / 예상 비용 (gpt-4.1-mini) ---")
    print(f"입력 토큰(전체): {u['input_tokens']:,}")
    print(f"  - 캐시 히트: {u['cache_read_tokens']:,}")
    print(f"  - 캐시 미적용: {non_cached_input:,}")
    print(f"출력 토큰: {u['output_tokens']:,}")
    print(f"예상 비용: 약 ${total_cost:.2f}")


await main()